## Bronze to Silver: Data Cleansing and Transformation

In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

#### Create Widgets

In [0]:
dbutils.widgets.text("catalog_name","ecommerce","Catalog Name")
dbutils.widgets.text("storage_account_name","rwecommerceadlsdevus001","Storage Account Name")
dbutils.widgets.text("container_name","ecom-raw-data")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

### Stream Bronze Table in a Dataframe

In [0]:
df = spark.readStream\
    .format("delta")\
    .table(f"{catalog_name}.bronze.brz_order_items")

In [0]:
dfs = spark.read \
    .format("delta") \
    .table(f"{catalog_name}.bronze.brz_order_items")

display(dfs.limit(20))

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,_rescued_data,ingest_timestamp,source_file
2025-09-03,2025-09-03T21:38:03.000Z,CUST000000196973,681661,1,2000000140087,1,INR,2182,7%,102,app,null,null,2026-05-21T17:30:35.673Z,abfss://ecom-raw-data@rwecommerceadlsdevus001.dfs.core.windows.net/order_items/landing/order_items_2025-09-03.csv
2025-09-03,2025-09-03T21:38:03.000Z,CUST000000196973,681661,2,2000000327167,1,INR,31445,20%,3030,app,null,null,2026-05-21T17:30:35.673Z,abfss://ecom-raw-data@rwecommerceadlsdevus001.dfs.core.windows.net/order_items/landing/order_items_2025-09-03.csv
2025-09-03,2025-09-03T07:54:07.000Z,CUST000000179098,681662,1,2000000430799,Two,SGD,699,8%,233,web,null,null,2026-05-21T17:30:35.673Z,abfss://ecom-raw-data@rwecommerceadlsdevus001.dfs.core.windows.net/order_items/landing/order_items_2025-09-03.csv
2025-09-03,2025-09-03T02:00:36.000Z,CUST000000272048,681663,1,2000000047300,1,GBP,98,19%,4,app,null,null,2026-05-21T17:30:35.673Z,abfss://ecom-raw-data@rwecommerceadlsdevus001.dfs.core.windows.net/order_items/landing/order_items_2025-09-03.csv
2025-09-03,2025-09-03T14:27:22.000Z,CUST000000248034,681664,1,2000000373096,1,GBP,12,16%,2,web,null,null,2026-05-21T17:30:35.673Z,abfss://ecom-raw-data@rwecommerceadlsdevus001.dfs.core.windows.net/order_items/landing/order_items_2025-09-03.csv
2025-09-03,2025-09-03T14:23:07.000Z,CUST000000025277,681665,1,2000000343181,1,INR,2074,4%,239,web,null,null,2026-05-21T17:30:35.673Z,abfss://ecom-raw-data@rwecommerceadlsdevus001.dfs.core.windows.net/order_items/landing/order_items_2025-09-03.csv
2025-09-03,2025-09-03T14:23:07.000Z,CUST000000025277,681665,2,2000000138251,1,INR,4344,13%,190,web,null,null,2026-05-21T17:30:35.673Z,abfss://ecom-raw-data@rwecommerceadlsdevus001.dfs.core.windows.net/order_items/landing/order_items_2025-09-03.csv
2025-09-03,2025-09-03T13:59:17.000Z,CUST000000105400,681666,1,2000000127897,4,USD,$64,0%,32,app,FEST20,null,2026-05-21T17:30:35.673Z,abfss://ecom-raw-data@rwecommerceadlsdevus001.dfs.core.windows.net/order_items/landing/order_items_2025-09-03.csv
2025-09-03,2025-09-03T13:59:17.000Z,CUST000000105400,681666,2,2000000191577,1,USD,$1031,14%,45,app,FEST20,null,2026-05-21T17:30:35.673Z,abfss://ecom-raw-data@rwecommerceadlsdevus001.dfs.core.windows.net/order_items/landing/order_items_2025-09-03.csv
2025-09-03,2025-09-03T13:59:17.000Z,CUST000000105400,681666,3,2000000482866,1,USD,$52,7%,9,app,FEST20,null,2026-05-21T17:30:35.673Z,abfss://ecom-raw-data@rwecommerceadlsdevus001.dfs.core.windows.net/order_items/landing/order_items_2025-09-03.csv


In [0]:
df = df.dropDuplicates(["order_id", "item_seq"])

# Transformation: Convert 'Two' → 2 and cast to Integer
df = df.withColumn(
    "quantity",
    F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int")
)

# Transformation : Remove any '$' or other symbols from unit_price, keep only numeric
df = df.withColumn(
    "unit_price",
    F.regexp_replace("unit_price", "[$]", "").cast("double")
)

# Transformation : Remove '%' from discount_pct and cast to double
df = df.withColumn(
    "discount_pct",
    F.regexp_replace("discount_pct", "%", "").cast("double")
)

# Transformation : coupon code processing (convert to lower)
df = df.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)

# Transformation : channel processing 
df = df.withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
    .when(F.col("channel") == "app", "Mobile")
    .otherwise(F.col("channel")),
)

#Transformation : Add processed time 
df = df.withColumn(
    "processed_time", F.current_timestamp()
)


In [0]:
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_order_items/"
dbutils.fs.mkdirs(silver_checkpoint_path)
print(silver_checkpoint_path)

abfss://ecom-raw-data@rwecommerceadlsdevus001.dfs.core.windows.net/checkpoint/silver/fact_order_items/


In [0]:
def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catalog_name}.silver.slv_order_items"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_id = batch_table.order_id AND silver_table.item_seq = batch_table.item_seq",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
# This line is running a Structured Streaming job that:
# - Reads incremental data from Bronze (df).
# - For each batch → applies upsert_to_silver (update if exists, insert if not).
# - Writes into a Silver Delta table with schema evolution enabled.
# - Uses checkpointing for recovery.
# - Runs in batch-like mode (once or availableNow), not continuous streaming.

df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()